# Clase 2 — RAG Avanzado

## Pre-retrieval y post-retrieval enhancements

**Objetivos de la clase:**
- Mejorar el pipeline naive de la Clase 1 con técnicas de **pre-retrieval** (mejor chunking) y **post-retrieval** (fusion retrieval, reranking, filtrado).
- Combinar búsqueda semántica (vectorial) con búsqueda por palabras clave (BM25) — *hybrid search*.
- Reordenar resultados con un modelo cross-encoder (reranking).
- Filtrar documentos irrelevantes antes de generar la respuesta.
- Comparar, sobre las mismas preguntas, el pipeline naive vs el avanzado.

📎 Material de referencia: `Clase_8_RAG.pdf` (slides "RAG Avanzado", "Mejoras en RAG", "RAG Modular", "Optimización del Pipeline RAG").

## Recordando la Clase 1

Construimos un pipeline RAG **naive**: chunking fijo → embeddings OpenAI → FAISS → top-`k` por similitud coseno → generación. Funciona, pero tiene límites (slide "Desafíos del RAG Tradicional"):

- **Comprensión superficial de la consulta**: solo similitud vectorial, puede fallar en captar matices.
- **Redundancia y ruido**: se entregan todos los chunks recuperados al LLM, sin filtrar.
- **Proceso lineal rígido**: "recuperar → generar", sin retroceder ni reevaluar.

Hoy vamos a atacar estos tres puntos.

In [1]:
# Instalar los paquetes necesarios
!pip install -q wikipedia python-dotenv==1.1.0 \
    langchain-community==0.3.25 langchain_openai==0.3.23 faiss-cpu==1.11.0 \
    rank_bm25==0.2.2 langchain-huggingface sentence-transformers tiktoken ipywidgets

# Evitar que transformers intente usar TensorFlow/Keras 3 (no lo necesitamos: el cross-encoder
# de reranking y los embeddings locales corren en PyTorch). Debe ejecutarse ANTES de importar
# sentence_transformers por primera vez — si ya lo importaste en esta sesión, reinicia el kernel/runtime.
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [21]:
import os

try:
    from google.colab import userdata
    from google.colab.userdata import SecretNotFoundError
except ModuleNotFoundError:
    userdata = None
    # Define a dummy SecretNotFoundError if not in Colab to avoid NameError
    class SecretNotFoundError(Exception):
        pass

def obtener_secret(nombre, prompt_text=None):
    if userdata is not None:
        try:
            valor = userdata.get(nombre)
            if valor:
                return valor
        except SecretNotFoundError:
            # Secret not found in userdata, proceed to next options
            pass

    valor = os.getenv(nombre)
    if valor:
        return valor

    return input(prompt_text or f"Ingresa {nombre}: ")

# OpenRouter se usa para el LLM (generación, grader, etc.) — crea tu key en openrouter.ai/keys
# Los embeddings de esta clase son locales (HuggingFace), así que no necesitamos OpenAI para nada.
os.environ["OPENROUTER_API_KEY"] = obtener_secret("OPENROUTER_API_KEY", "OpenRouter API key: ")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "deepseek/deepseek-v4-flash"  # cámbialo por cualquier modelo de openrouter.ai/models

In [7]:
from langchain_community.document_loaders import WikipediaLoader

WIKI_QUERY = "Cambio climático en Chile"

documento = WikipediaLoader(
    query=WIKI_QUERY, lang="es", load_max_docs=1, doc_content_chars_max=40000
).load()

print(f"Artículo cargado: '{documento[0].metadata.get('title')}' ({len(documento[0].page_content)} caracteres)")


Artículo cargado: 'Cambio climático en Chile' (36503 caracteres)


## 1. Reconstruir el pipeline base (recordatorio de Tutorial 1)

In [8]:
import ipywidgets as widgets
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def dividir_en_chunks(documento, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    chunks = splitter.split_documents(documento)
    for c in chunks:
        c.page_content = c.page_content.replace('\t', ' ')
    return chunks

def crear_llm(model=None, **kwargs):
    """ChatOpenAI apuntando a OpenRouter en vez de a la API de OpenAI directamente."""
    return ChatOpenAI(
        model=model or OPENROUTER_MODEL,
        base_url=OPENROUTER_BASE_URL,
        api_key=os.environ["OPENROUTER_API_KEY"],
        **kwargs,
    )

llm = crear_llm(temperature=0)

# Embeddings locales: corren gratis en tu CPU, sin depender de una API externa.
embeddings = HuggingFaceEmbeddings(
    # model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    model_name = "sentence-transformers/LaBSE",
    encode_kwargs={"batch_size": 32, "show_progress_bar": True},
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 2. Pre-retrieval: ¿el chunk_size calza con el modelo de embeddings?

Antes de indexar, conviene revisar si el largo de nuestros chunks (en **tokens**, no caracteres) es razonable para el modelo de embeddings que usaremos. Chunks demasiado largos se truncan silenciosamente; chunks demasiado cortos pierden contexto.

In [9]:
from sentence_transformers import SentenceTransformer

# Load the SentenceTransformer model directly to get its max_seq_length
# The embeddings object's client attribute might not be consistently available or exposed
temp_model = SentenceTransformer(embeddings.model_name)
LIMITE_MODELO_LOCAL = temp_model.max_seq_length
print(f"max_seq_length de {embeddings.model_name}: {LIMITE_MODELO_LOCAL} tokens")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

max_seq_length de sentence-transformers/LaBSE: 256 tokens


In [10]:
import tiktoken
import matplotlib.pyplot as plt

encoding = tiktoken.get_encoding("cl100k_base")

@widgets.interact(chunk_size=widgets.IntSlider(value=1000, min=100, max=2000, step=100))
def histograma_chunks(chunk_size):
    chunks_tmp = dividir_en_chunks(documento, chunk_size=chunk_size, chunk_overlap=int(chunk_size * 0.2))
    largos = [len(encoding.encode(c.page_content)) for c in chunks_tmp]
    plt.figure(figsize=(7, 4))
    plt.hist(largos, bins=20)
    plt.axvline(LIMITE_MODELO_LOCAL, color="red", linestyle="--", label=f"límite del modelo local ({LIMITE_MODELO_LOCAL} tokens)")
    plt.title(f"Distribución de largo de chunks en tokens (chunk_size={chunk_size} caracteres)")
    plt.xlabel("tokens (aprox., contados con el tokenizer de OpenAI)"); plt.ylabel("cantidad de chunks"); plt.legend()
    plt.show()
    print(f"{len(chunks_tmp)} chunks generados")

interactive(children=(IntSlider(value=1000, description='chunk_size', max=2000, min=100, step=100), Output()),…

In [11]:
# Fijamos la configuración para el resto de la clase.
# chunk_size más chico: el modelo local  trunca a 256 tokens,
# muy por debajo del límite de OpenAI (8191) — con chunks de 800 caracteres nos mantenemos
# cómodamente bajo ese límite (ver histograma de la sección anterior).
CHUNK_SIZE, CHUNK_OVERLAP = 800, 100
chunks = dividir_en_chunks(documento, CHUNK_SIZE, CHUNK_OVERLAP)

# Le damos un id explícito a cada chunk: lo vamos a necesitar para alinear
# los puntajes de BM25 y de similitud vectorial en el fusion retrieval.
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/LaBSE",
    encode_kwargs={"batch_size": 32},
    show_progress=True,
)

vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"{len(chunks)} chunks indexados en FAISS.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

77 chunks indexados en FAISS.


## 3. Fusion Retrieval: combinar BM25 + búsqueda vectorial

*Hybrid search* (slide "Optimización del Pipeline RAG") combina:

- **BM25**: ranking por coincidencia de palabras clave — bueno para términos exactos, nombres propios, siglas.
- **Búsqueda vectorial**: ranking por similitud semántica — bueno para paráfrasis y sinónimos.

El parámetro `alpha` controla el peso relativo: `alpha=1` es 100% vectorial, `alpha=0` es 100% BM25.

In [12]:
from rank_bm25 import BM25Okapi
import numpy as np

def crear_indice_bm25(chunks):
    tokenized = [c.page_content.split() for c in chunks]
    return BM25Okapi(tokenized)

bm25 = crear_indice_bm25(chunks)

def fusion_retrieval(vectorstore, bm25, chunks, query, k=5, alpha=0.5):
    """Combina BM25 y similitud vectorial, alineando los puntajes por chunk_id."""
    epsilon = 1e-8
    n = len(chunks)

    bm25_scores = np.array(bm25.get_scores(query.split()))
    bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + epsilon)

    vector_results = vectorstore.similarity_search_with_score(query, k=n)
    # similarity_search_with_score en FAISS devuelve distancia L2 (menor = más similar)
    vector_scores_por_id = np.zeros(n)
    for doc, score in vector_results:
        vector_scores_por_id[doc.metadata["chunk_id"]] = score
    vector_scores_por_id = 1 - (
        (vector_scores_por_id - vector_scores_por_id.min())
        / (vector_scores_por_id.max() - vector_scores_por_id.min() + epsilon)
    )

    combinado = alpha * vector_scores_por_id + (1 - alpha) * bm25_scores
    top_idx = np.argsort(combinado)[::-1][:k]
    return [chunks[i] for i in top_idx], combinado[top_idx]

In [13]:
import pandas as pd

@widgets.interact_manual(
    pregunta=widgets.Textarea(value="¿Qué establece la Ley Marco de Cambio Climático de Chile?",
                              description='Pregunta:', layout=widgets.Layout(width='auto')),
    alpha=widgets.FloatSlider(value=0.5, min=0, max=1, step=0.1),
    k=widgets.IntSlider(value=5, min=1, max=10),
)
def probar_fusion(pregunta, alpha, k):
    docs, scores = fusion_retrieval(vectorstore, bm25, chunks, pregunta, k=k, alpha=alpha)
    df = pd.DataFrame({
        "score_combinado": scores.round(3),
        "preview": [d.page_content[:500].replace("\n", " ") + "..." for d in docs],
    })
    pd.set_option('display.max_colwidth', None)
    display(df)

interactive(children=(Textarea(value='¿Qué establece la Ley Marco de Cambio Climático de Chile?', description=…

## 4. Post-retrieval: Reranking con un cross-encoder

Un *bi-encoder* (como el que usamos para embeddings) codifica pregunta y documento **por separado** — rápido, pero menos preciso. Un **cross-encoder** los codifica **juntos**, por lo que puede captar mejor la relación entre ambos — más preciso, pero más lento. Por eso se usa como segunda pasada (*reranking*) sobre un conjunto ya reducido de candidatos (slide "Post-Retrieval Enhancements: Reranking").

In [14]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(pregunta, docs):
    pares = [(pregunta, d.page_content) for d in docs]
    scores = reranker.predict(pares)
    orden = np.argsort(scores)[::-1]
    return [docs[i] for i in orden], scores[orden]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [15]:
@widgets.interact_manual(
    pregunta=widgets.Textarea(value="¿Cómo afecta el cambio climático a la biodiversidad en Chile?",
                              description='Pregunta:', layout=widgets.Layout(width='auto')),
    alpha=widgets.FloatSlider(value=0.5, min=0, max=1, step=0.1),
    k=widgets.IntSlider(value=5, min=1, max=10),
)
def probar_reranking(pregunta):
    docs, _ = fusion_retrieval(vectorstore, bm25, chunks, pregunta, k=8, alpha=0.5)
    docs_rerankeados, scores = rerank(pregunta, docs)

    print("Orden ANTES del reranking (fusion retrieval):")
    for d in docs[:3]:
        print("-", d.page_content[:120].replace("\n", " "), "...")

    print("\nOrden DESPUÉS del reranking (cross-encoder):")
    for d, s in zip(docs_rerankeados[:3], scores[:3]):
        print(f"- (score={s:.2f})", d.page_content[:120].replace("\n", " "), "...")

interactive(children=(Textarea(value='¿Cómo afecta el cambio climático a la biodiversidad en Chile?', descript…

## 5. Post-retrieval: filtrar documentos irrelevantes con un LLM

Incluso después de reranking, algunos chunks pueden no aportar nada a la respuesta ("Filtering" en la slide "Mejoras en RAG"). Usamos un LLM como *grader* binario: para cada chunk, ¿es relevante para la pregunta o no?

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class GradeDocuments(BaseModel):
    binary_score: str = Field(description="'yes' si el documento es relevante para la pregunta, 'no' si no lo es")

# method="function_calling" a propósito: el modo automático ("json_schema") de with_structured_output
# asume que estás hablando directo con la API de OpenAI y en OpenRouter puede fallar con errores de
# autenticación raros (toma otro camino de cliente internamente). function_calling usa tool-calling
# normal, que es el mismo camino que ya probamos que funciona con OpenRouter.
grader_llm = llm.with_structured_output(GradeDocuments, method="function_calling")

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un evaluador que determina si un documento recuperado es relevante para responder una "
               "pregunta de un usuario. No hace falta ser muy estricto — el objetivo es filtrar solo los "
               "resultados claramente erróneos. Da un puntaje binario 'yes' o 'no'."),
    ("human", "Documento recuperado:\n\n{document}\n\nPregunta del usuario: {question}"),
])
retrieval_grader = grade_prompt | grader_llm

def filtrar_relevantes(pregunta, docs):
    relevantes = []
    for d in docs:
        res = retrieval_grader.invoke({"question": pregunta, "document": d.page_content})
        if res.binary_score == "yes":
            relevantes.append(d)
    return relevantes

In [17]:
@widgets.interact_manual(
    pregunta=widgets.Textarea(value="¿Cómo afecta el cambio climático a la biodiversidad en Chile?",
                              description='Pregunta:', layout=widgets.Layout(width='auto')),
    alpha=widgets.FloatSlider(value=0.5, min=0, max=1, step=0.1),
    k=widgets.IntSlider(value=5, min=1, max=10),
)
def probar_filtro(pregunta):
    docs, _ = fusion_retrieval(vectorstore, bm25, chunks, pregunta, k=6, alpha=0.5)
    docs_rerankeados, _ = rerank(pregunta, docs)
    relevantes = filtrar_relevantes(pregunta, docs_rerankeados)
    print(f"{len(docs_rerankeados)} documentos recuperados -> {len(relevantes)} marcados como relevantes")
    for d in relevantes:
        print("-", d.page_content[:200].replace("\n", " "), "...")

interactive(children=(Text(value='¿Cómo afecta el cambio climático a la biodiversidad en Chile?', description=…

## 6. Pipeline avanzado completo vs Tutorial 1

Juntamos todo: fusion retrieval → reranking → filtrado → generación. Comparemos, sobre la misma pregunta, el pipeline **naive** (Tutorial 1) contra el **avanzado** (hoy).

In [18]:
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde ÚNICAMENTE con base en el contexto entregado. Si falta información dilo explícitamente."),
    ("human", "Contexto:\n{contexto}\n\nPregunta: {pregunta}"),
])
rag_chain = rag_prompt | llm | StrOutputParser()

def pipeline_naive(pregunta, k=3):
    docs = vectorstore.as_retriever(search_kwargs={"k": k}).invoke(pregunta)
    contexto = "\n\n".join(d.page_content for d in docs)
    return rag_chain.invoke({"contexto": contexto, "pregunta": pregunta})

def pipeline_avanzado(pregunta, k=8, alpha=0.5, k_final=5):
    docs, _ = fusion_retrieval(vectorstore, bm25, chunks, pregunta, k=k, alpha=alpha)
    docs, _ = rerank(pregunta, docs)
    docs = filtrar_relevantes(pregunta, docs[:k_final])
    contexto = "\n\n".join(d.page_content for d in docs)
    return rag_chain.invoke({"contexto": contexto, "pregunta": pregunta})

In [20]:
@widgets.interact_manual(
    pregunta=widgets.Textarea(value="¿Cuánto han aumentado las emisiones de gases de efecto invernadero en Chile desde 1990?",
                              description='Pregunta:', layout=widgets.Layout(width='auto')),
    alpha=widgets.FloatSlider(value=0.5, min=0, max=1, step=0.1),
    k=widgets.IntSlider(value=5, min=1, max=10),
)
def comparar_naive_vs_avanzado(pregunta):
    print("=" * 20, "PIPELINE CLASE 1 (naive)", "=" * 20)
    print(pipeline_naive(pregunta))
    print()
    print("=" * 20, "PIPELINE CLASE 2 (avanzado)", "=" * 20)
    print(pipeline_avanzado(pregunta))

interactive(children=(Textarea(value='¿Cuánto han aumentado las emisiones de gases de efecto invernadero en Ch…

## 7. Un paso más allá: RAG Modular

El esquema "recuperar → generar" es rígido. **RAG Modular** (slide "Introducción a RAG Modular") propone descomponer el pipeline en módulos independientes — `Rewrite`, `Retrieve`, `Rerank`, `Read`, `Fusion`, `Memory`, etc. — que se pueden **reconfigurar según la tarea**, como piezas de LEGO.

Hoy ya construimos, "a mano", varios de estos módulos: reescritura implícita (fusion retrieval), rerank, y read (generación).  


## Ejercicios propuestos

1. **Ajustar alpha**
   - Prueba `alpha=0` (solo BM25) y `alpha=1` (solo vectorial) con una pregunta que use términos técnicos exactos del documento (ej. "Ley Marco de Cambio Climático", "CONAF", "Fondo Verde del Clima"). ¿Cuál rinde mejor para ese tipo de preguntas?

2. **Con y sin reranking**
   - Compara la respuesta final del pipeline avanzado con y sin el paso de reranking (puedes comentar esa línea en `pipeline_avanzado`). ¿Cambia la calidad de la respuesta?

3. **Con y sin filtrado**
   - Haz una pregunta ambigua o parcialmente fuera de tema. ¿El filtrado de relevancia logra descartar chunks que no aportan?

## 🔜 Próxima clase

Ahora tenemos **dos** pipelines (naive y avanzado) y una intuición de que el segundo es "mejor" — pero solo la hemos evaluado mirando respuestas a ojo. En la **Clase 2 — Evaluación de RAG** vamos a medir esto de forma rigurosa: métricas de retrieval (Recall@k, MRR), métricas de generación, y métricas específicas de RAG (faithfulness, answer relevancy, context precision/recall).